In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nfl_data_py as nfl
import sys

from sklearn import metrics
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.model_selection import ShuffleSplit
from sklearn.model_selection import KFold
from sklearn.model_selection import ParameterGrid
from sklearn.model_selection import ParameterSampler
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import plot_tree
import xgboost as xgb
import shap
import optuna

sys.path.append("../src")
from features.feature_engineering import main as import_data

In [ ]:
data = import_data(2010, 2024, 1/3.0, 5)
data.head()

,game_id,season,week,home_team,away_team,result,home_line,over_under,roof,surface,...,yds_adv,pass_cmp_adv,sack_yards_adv,third_down_attempts_adv,ydsnet_mean_adv_home,ydsnet_mean_adv_away,tackled_for_loss_sum_adv_home,tackled_for_loss_sum_adv_away,yards_after_catch_sum_adv_home,yards_after_catch_sum_adv_away
0,20110908_nor_gnb,2011,1,gnb,nor,8.0,-5.0,48.0,outdoors,grass,...,-13.147059,-4.443137,1.545098,-0.552941,2.704534,6.686245,-0.107843,0.000000,8.970588,21.147059
1,20110911_phi_ram,2011,1,ram,phi,-18.0,4.0,44.5,indoors,turf,...,-56.200980,0.318627,-2.774510,0.889706,-1.969111,1.393940,-0.446078,-0.855392,6.737745,6.928922
2,20110911_clt_htx,2011,1,htx,clt,27.0,-9.0,44.0,outdoors,grass,...,6.612745,-3.144608,5.612745,-0.509804,1.619553,1.924899,0.213235,0.017157,15.774510,-6.669118
3,20110911_oti_jax,2011,1,jax,oti,2.0,1.5,38.0,outdoors,grass,...,26.083333,0.750000,5.000000,-0.041667,-1.492984,-6.736981,-0.958333,0.333333,-21.541667,-32.250000
4,20110911_pit_rav,2011,1,rav,pit,28.0,-1.5,37.0,outdoors,turf,...,-16.292398,0.575049,0.953216,-0.130604,2.807296,2.215047,0.325536,0.561404,8.651072,3.403509


In [22]:
def prepare_data(
    data: pd.DataFrame,
    train_seasons: list[int],
    test_seasons: list[int],
    features: list[str],
) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame, pd.Series]:
    """
    Partition a dataset into training and testing sets, dropping rows with missing 
    feature values.

    Args:
        data: Input dataset containing all seasons and target column 'result'.
        train_seasons: List of season values to include in the training set.
        test_seasons: List of season values to include in the testing set.
        features: List of feature column names to use as predictors.

    Returns:
        tuple: Training feature matrix, training target vector, testing feature matrix
            testing target vector.
    """
    df_train = data[data["season"].isin(train_seasons)].dropna(subset=features)
    df_test = data[data["season"].isin(test_seasons)].dropna(subset=features)

    X_train = df_train[features]
    y_train = df_train["result"]
    X_test = df_test[features]
    y_test = df_test["result"]

    return X_train, y_train, X_test, y_test

In [27]:
train_seasons = [2011, 2012, 2013, 2014]
test_seasons = [2015]
features = [col for col in data.columns if col not in ["game_id", "season", "week", "home_team", "away_team", "result", "roof", "surface", "attendance"]]

X_train, y_train, X_test, y_test = prepare_data(data, train_seasons, test_seasons, features)